# Quick Baseline for Colab

This notebook is a compact version of the Day 1 pipeline: load data, preprocess numeric descriptors, train a simple model, validate it and create a submission.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.ensemble import ExtraTreesRegressor
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import KFold
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42
TARGETS = ['IC50, mM', 'CC50, mM', 'SI']
SUBMISSION_TARGETS = ['IC50', 'CC50', 'SI']

Upload `train.csv`, `test.csv` and `sample_submission.csv` to the Colab working directory before running the next cell.

In [ ]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
sample_submission = pd.read_csv('sample_submission.csv')

feature_cols = [c for c in train.columns if c not in ['index', *TARGETS]]
X = train[feature_cols]
y = train[TARGETS]
X_test = test[feature_cols]

print(train.shape, test.shape, sample_submission.shape)
print('features:', len(feature_cols))
print('missing train:', int(X.isna().sum().sum()))
print('missing test:', int(X_test.isna().sum().sum()))

In [ ]:
def rmse(y_true, y_pred):
    return root_mean_squared_error(y_true, y_pred)

def score_targets(y_true, y_pred):
    scores = {}
    for i, target in enumerate(TARGETS):
        scores[target] = rmse(y_true[target], y_pred[:, i])
    target_scores = [scores[t] for t in TARGETS]
    scores['score'] = np.mean(target_scores)
    return scores

In [ ]:
model = Pipeline([
    ('preprocess', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('constant_filter', VarianceThreshold(0.0)),
    ])),
    ('model', MultiOutputRegressor(
        ExtraTreesRegressor(
            n_estimators=300,
            min_samples_leaf=2,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        n_jobs=-1,
    )),
])

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
oof = np.zeros((len(X), len(TARGETS)))
fold_scores = []

for fold, (tr_idx, val_idx) in enumerate(cv.split(X), 1):
    model.fit(X.iloc[tr_idx], y.iloc[tr_idx])
    pred = np.clip(model.predict(X.iloc[val_idx]), 0, None)
    oof[val_idx] = pred
    scores = score_targets(y.iloc[val_idx], pred)
    scores['fold'] = fold
    fold_scores.append(scores)

report = pd.DataFrame(fold_scores)
overall = score_targets(y, oof)
display(report)
print('overall:', overall)

In [ ]:
model.fit(X, y)
test_pred = np.clip(model.predict(X_test), 0, None)

submission = sample_submission[['index']].copy()
for i, col in enumerate(SUBMISSION_TARGETS):
    submission[col] = test_pred[:, i]

submission.to_csv('baseline_submission.csv', index=False)
submission.head()